# N5 — FF49 Baseline Peer Lists (M0)

Builds the M0 baseline peer lists using Fama-French 49 industry classification.
All firms in the same FF49 industry and same fyear are assigned as peers with
equal weight (similarity_score = 1.0, rank = 1 for all).

This is the direct methodological equivalent of how analysts use industry
classifications in practice — every firm in the same industry is a valid peer.

**Design decisions:**
- Peers restricted to same fyear (consistent with M1/M2/M3)
- All industry peers receive equal weight: similarity_score = 1.0, rank = 1
- No kNN — M0 is purely membership-based
- k varies by industry size (Banking ~50+, Tobacco ~5) — this is intentional

**Input:** `panel_clean.parquet` (N1)
**Output:** `peers_m0.parquet` — schema: `focal_tic | focal_fyear | peer_tic | rank | similarity_score | model`


In [ ]:
# imports & config
import sys
from pathlib import Path

notebook_dir = Path('__file__').parent if '__file__' in dir() else Path.cwd()
repo_root = next(
    (p for p in [notebook_dir, *notebook_dir.parents] if (p / 'config.py').exists()),
    None
)
if repo_root is None:
    raise FileNotFoundError('config.py not found — check repo structure')
sys.path.insert(0, str(repo_root))

from config import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Config loaded.")
print(f"  PANEL_CLEAN : {PANEL_CLEAN}")
print(f"  PEERS_M0    : {PEERS_M0}")
print(f"  YEARS       : {YEARS}")


In [ ]:
# declare I/O
INPUTS  = [PANEL_CLEAN]
OUTPUTS = [PEERS_M0]
# PURPOSE : Build M0 FF49 baseline peer lists — all same-industry firms per fyear
# RUNTIME : ~2 min
# DEPENDS : panel_clean.parquet (N1)


## 1. Load Panel

In [ ]:
# load panel and restrict to evaluation sample
df_full = pd.read_parquet(PANEL_CLEAN, columns=[
    'gvkey', 'tic', 'fyear', 'ff49_num', 'ff49_abbr', 'industry'
])
print(f"Full panel: {df_full.shape[0]:,} rows | {df_full['tic'].nunique():,} unique tickers")

# Build evaluation sample — firms with valid Gemini summaries
INVALID_FLAGS = {'ERROR', 'INSUFFICIENT_DATA', 'ERROR_EXTRACTING_TEXT'}
eval_tickers = set()
for yr in YEARS:
    path = SUMMARIES_FILES[yr]
    if not path.exists():
        print(f"  WARNING: {path} not found")
        continue
    df_s = pd.read_csv(path)
    valid = df_s[
        df_s['business_description'].notna() &
        ~df_s['business_description'].isin(INVALID_FLAGS) &
        (df_s['business_description'].str.len() > 50)
    ]
    for _, row in valid.iterrows():
        eval_tickers.add((row['tic'], int(row['fyear'])))

print(f"Evaluation sample: {len(eval_tickers):,} firm-years | "
      f"{len(set(t for t, _ in eval_tickers)):,} unique tickers")

# Restrict panel to evaluation sample only
df = df_full[
    df_full.apply(lambda r: (r['tic'], int(r['fyear'])) in eval_tickers, axis=1)
].copy()

print(f"Restricted panel : {df.shape[0]:,} rows | {df['tic'].nunique():,} unique tickers")
print(f"Years            : {sorted(df['fyear'].unique())}")

## 2. Industry Size Distribution

Understanding the peer group sizes is important for interpreting M0 results.
Large industries (Banking, Trading) will have many peers; small industries
(Tobacco, Coal) will have very few. This variability is a known limitation
of the FF49 baseline.


In [ ]:
# industry size distribution per fyear
print("Peer group sizes by industry (averaged across years):")
print()

sizes = (df.groupby(['fyear', 'ff49_num', 'industry'])['tic']
           .count()
           .reset_index()
           .rename(columns={'tic': 'n_peers'}))

avg_sizes = (sizes.groupby(['ff49_num', 'industry'])['n_peers']
                  .agg(['mean', 'min', 'max'])
                  .reset_index()
                  .sort_values('mean', ascending=False))

print(avg_sizes.to_string(index=False))
print()
print(f"Overall: mean={sizes['n_peers'].mean():.0f} peers | "
      f"min={sizes['n_peers'].min()} | max={sizes['n_peers'].max()}")


In [ ]:
# visualise industry size distribution
fig, ax = plt.subplots(figsize=(12, 7))

avg_plot = avg_sizes.sort_values('mean', ascending=True)
y_pos = range(len(avg_plot))

ax.barh(y_pos, avg_plot['mean'], color='#4C72B0', alpha=0.8, label='Mean')
ax.set_yticks(y_pos)
ax.set_yticklabels(avg_plot['industry'], fontsize=8)
ax.set_xlabel("Average number of peers per fyear")
ax.set_title("M0 FF49 Baseline — Peer Group Size by Industry", fontsize=11)
ax.axvline(avg_plot['mean'].mean(), color='red', linewidth=1,
           linestyle='--', label=f"Overall mean ({avg_plot['mean'].mean():.0f})")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / "n5_m0_industry_sizes.pdf", dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES / 'n5_m0_industry_sizes.pdf'}")


## 3. Build M0 Peer Lists

For each focal firm, all other firms in the same FF49 industry and same fyear
are assigned as peers with equal weight (similarity_score = 1.0, rank = 1).


In [ ]:
# build M0 peer lists
print("Building M0 FF49 baseline peer lists...")
print()

all_peers = []

for yr in YEARS:
    df_year = df[df['fyear'] == yr].dropna(subset=['tic']).copy()
    n_firms = len(df_year)
    records = []

    # Group by FF49 industry
    for ff49_num, group in df_year.groupby('ff49_num'):
        tickers = group['tic'].values
        n_peers = len(tickers)

        if n_peers < 2:
            # Only one firm in this industry-year — no peers possible
            continue

        for focal_tic in tickers:
            # All other firms in same industry = peers
            peer_tickers = tickers[tickers != focal_tic]
            for peer_tic in peer_tickers:
                records.append({
                    'focal_tic'       : focal_tic,
                    'focal_fyear'     : yr,
                    'peer_tic'        : peer_tic,
                    'rank'            : 1,          # equal weight — all rank 1
                    'similarity_score': 1.0,        # equal weight
                    'model'           : 'M0_FF49',
                })

    peers_yr = pd.DataFrame(records)
    all_peers.append(peers_yr)

    n_records = len(peers_yr)
    avg_peers = n_records / n_firms if n_firms > 0 else 0
    print(f"  {yr}: {n_firms:,} firms → {n_records:,} peer records "
          f"(avg {avg_peers:.0f} peers per firm)")

peers_m0 = pd.concat(all_peers, ignore_index=True)
print()
print(f"Total peer records: {len(peers_m0):,}")


## 4. Validation

In [ ]:
# schema validation
print("Schema check:")
for col in PEER_SCHEMA:
    present = col in peers_m0.columns
    dtype   = peers_m0[col].dtype if present else "MISSING"
    print(f"  {col:<20} {'✓' if present else '✗'}  {dtype}")

print()
print("Null check:")
print(peers_m0.isnull().sum().to_string())

print()
# Verify rank is always 1
assert (peers_m0['rank'] == 1).all(), "ERROR: rank should be 1 for all M0 peers"
assert (peers_m0['similarity_score'] == 1.0).all(), "ERROR: similarity_score should be 1.0"
assert (peers_m0['model'] == 'M0_FF49').all(), "ERROR: model label incorrect"
print("All assertions passed ✓")
print(f"  rank == 1 for all records")
print(f"  similarity_score == 1.0 for all records")
print(f"  model == 'M0_FF49' for all records")


In [ ]:
# peer count distribution
peer_counts = (peers_m0.groupby(['focal_tic', 'focal_fyear'])['peer_tic']
                        .count()
                        .reset_index()
                        .rename(columns={'peer_tic': 'n_peers'}))

print("Peer count distribution across focal firm-years:")
print(f"  Mean    : {peer_counts['n_peers'].mean():.1f}")
print(f"  Median  : {peer_counts['n_peers'].median():.0f}")
print(f"  Min     : {peer_counts['n_peers'].min()}")
print(f"  Max     : {peer_counts['n_peers'].max()}")
print(f"  Std     : {peer_counts['n_peers'].std():.1f}")
print()

# Firms with very few peers (potential concern for evaluation)
few_peers = peer_counts[peer_counts['n_peers'] < 5]
print(f"Firm-years with <5 peers: {len(few_peers):,} "
      f"({len(few_peers)/len(peer_counts)*100:.1f}%)")
if len(few_peers) > 0:
    print("  High variance in MdAPE expected for these firm-years.")


In [ ]:
# spot check: verify peer symmetry
# If A is a peer of B, B must be a peer of A
print("Symmetry check (sample)...")
sample_focal = peers_m0['focal_tic'].iloc[0]
sample_year  = peers_m0['focal_fyear'].iloc[0]

focal_peers = set(peers_m0[
    (peers_m0['focal_tic'] == sample_focal) &
    (peers_m0['focal_fyear'] == sample_year)
]['peer_tic'])

# Check reciprocal
for peer in list(focal_peers)[:3]:
    peer_of_peer = set(peers_m0[
        (peers_m0['focal_tic'] == peer) &
        (peers_m0['focal_fyear'] == sample_year)
    ]['peer_tic'])
    assert sample_focal in peer_of_peer, f"Symmetry broken: {peer} → {sample_focal} missing"

print(f"  Symmetry verified for {sample_focal} (FY{sample_year}) ✓")
print(f"  Has {len(focal_peers)} industry peers")


## 5. Industry Coverage Diagnostics

In [ ]:
# firms with no peers (singleton industries)
all_focal = set(zip(df['tic'], df['fyear']))
has_peers = set(zip(peers_m0['focal_tic'], peers_m0['focal_fyear']))
no_peers  = all_focal - has_peers

print(f"Firm-years with NO peers (singleton in their industry-year): {len(no_peers):,}")
if no_peers:
    df_no_peers = pd.DataFrame(no_peers, columns=['tic', 'fyear'])
    df_no_peers = df_no_peers.merge(
        df[['tic', 'fyear', 'ff49_abbr', 'industry']], on=['tic', 'fyear'], how='left'
    )
    print()
    print(df_no_peers.groupby(['ff49_abbr', 'industry'])
                     .size()
                     .reset_index(name='count')
                     .sort_values('count', ascending=False)
                     .to_string(index=False))
    print()
    print("NOTE: These firm-years will be excluded from M0 evaluation in N9.")
    print("      They will still appear in M1/M2/M3 evaluation.")


## 6. Save

In [ ]:
# save peers_m0.parquet
peers_m0.to_parquet(PEERS_M0, index=False)

print(f"Saved : {PEERS_M0}")
print(f"Shape : {peers_m0.shape[0]:,} rows × {peers_m0.shape[1]} columns")
print()
print("Peer records per year:")
print(peers_m0.groupby('focal_fyear')['focal_tic'].count().to_string())
print()
print(f"Unique focal firms : {peers_m0['focal_tic'].nunique():,}")
print(f"Unique peer firms  : {peers_m0['peer_tic'].nunique():,}")


In [ ]:
# final summary
print("=" * 60)
print("N5 COMPLETE — M0 FF49 BASELINE SUMMARY")
print("=" * 60)
print(f"  Model          : M0_FF49")
print(f"  Method         : FF49 industry membership")
print(f"  Peer weight    : Equal (rank=1, similarity=1.0 for all)")
print(f"  Scope          : Same fyear only")
print(f"  Years          : {YEARS}")
print(f"  Total records  : {len(peers_m0):,}")
print()
print(f"  Key design note:")
print(f"  Peer count varies by industry size — this is intentional.")
print(f"  In N9 evaluation, MdAPE is computed using ALL available")
print(f"  industry peers, not a fixed k. This matches how analysts")
print(f"  use industry classifications in practice.")
print(f"  (Bhojraj & Lee 2002)")
print()
print(f"  Output: {PEERS_M0}")
print()
print("  Next: N6 (M1 financial kNN) — peers_m0.parquet consumed by N10 / N11 evaluation")
